In [13]:
import ast
import subprocess
import sys
from flask import Flask, request, render_template_string
import pandas as pd
import numpy as np
from neo4j import GraphDatabase
from neo4j.exceptions import ServiceUnavailable
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import partial
import torch
import time
from collections import defaultdict
import re

# Auto-install required packages
def install_packages():
    required = [
        'flask', 'pandas', 'numpy', 'neo4j', 'sentence-transformers', 
        'scikit-learn', 'transformers', 'torch', 'accelerate'
    ]
    for package in required:
        try:
            __import__(package)
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])

install_packages()

#########################################
# 1. Neo4j Connection and Interaction Agent (Optimized)
#########################################
class Neo4jAgent:
    def __init__(self, uri, user, password):
        self.uri = uri
        self.user = user
        self.password = password
        self.driver = None
        self.connect()
        self.batch_size = 5000  # Larger batch size for better performance
        self.cache = {}  # Cache for frequent queries
        
    def connect(self):
        try:
            self.driver = GraphDatabase.driver(
                self.uri, 
                auth=(self.user, self.password), 
                max_connection_lifetime=3600,
                max_connection_pool_size=20  # Increased pool size
            )
            with self.driver.session() as session:
                session.run("RETURN 1")
            print("Connected to Neo4j successfully.")
        except ServiceUnavailable as e:
            print("Neo4j connection error:", e)
            exit(1)
    
    def execute_write_batch(self, query, parameters=None):
        with self.driver.session() as session:
            return session.run(query, parameters).consume()
    
    def run_query(self, query, parameters=None, use_cache=False):
        cache_key = (query, frozenset(parameters.items()) if parameters else None)
        if use_cache and cache_key in self.cache:
            return self.cache[cache_key]
            
        with self.driver.session() as session:
            try:
                result = list(session.run(query, parameters or {}))
                if use_cache:
                    self.cache[cache_key] = result
                return result
            except Exception as e:
                print(f"Error running query: {e}")
                return []
    
    def get_node_count(self):
        result = self.run_query("MATCH (n) RETURN count(n) as count")
        return result[0]["count"] if result else 0
    
    def batch_create_nodes(self, label, id_field, data):
        query = f"""
        UNWIND $batch AS item
        MERGE (n:{label} {{{id_field}: item.id}})
        SET n += item.props
        """
        
        batches = []
        for i in range(0, len(data), self.batch_size):
            batch = data[i:i + self.batch_size]
            batches.append({
                "batch": [{
                    "id": item["props"].get(id_field) or item["props"].get(id_field.lower()) or item["props"].get('user_id') or item["props"].get('User_ID'),
                    "props": item["props"]
                } for item in batch]
            })
        
        with ThreadPoolExecutor() as executor:
            futures = [executor.submit(self.execute_write_batch, query, params) for params in batches]
            for future in as_completed(futures):
                future.result()  # Wait for completion
    
    def batch_create_relationships(self, label_from, key_from, rel_type, label_to, key_to, data):
        query = f"""
        UNWIND $batch AS item
        MATCH (a:{label_from} {{{key_from}: item.value_from}})
        MATCH (b:{label_to} {{{key_to}: item.value_to}})
        MERGE (a)-[r:{rel_type}]->(b)
        """
        
        batches = []
        for i in range(0, len(data), self.batch_size):
            batches.append({"batch": data[i:i + self.batch_size]})
        
        with ThreadPoolExecutor() as executor:
            futures = [executor.submit(self.execute_write_batch, query, params) for params in batches]
            for future in as_completed(futures):
                future.result()

#########################################
# 2. Graph Builder Agent (Optimized)
#########################################
class GraphBuilderAgent:
    def __init__(self, neo4j_agent, csv_paths):
        self.neo4j = neo4j_agent
        self.csv_paths = csv_paths
        self.executor = ThreadPoolExecutor(max_workers=8)  # More workers for parallel processing
    
    def process_csv(self, name, path):
        try:
            # Use low_memory=False to avoid mixed type warnings
            df = pd.read_csv(path, low_memory=False)
            print(f"Successfully loaded {path} with {len(df)} rows")
            return name, df
        except Exception as e:
            print(f"Error loading {path}: {e}")
            return name, pd.DataFrame()
    
    def build_graph(self):
        print("Starting graph build...")
        start_time = time.time()
        
        # Parallel CSV loading with progress tracking
        print("Loading CSV files...")
        csv_load_start = time.time()
        futures = {self.executor.submit(self.process_csv, name, path): name for name, path in self.csv_paths.items()}
        
        data = {}
        for future in as_completed(futures):
            name, df = future.result()
            data[name] = df
        print(f"CSV loading completed in {time.time() - csv_load_start:.2f} seconds")
        
        # Node creation configuration
        node_config = [
            ('cities', 'City', 'city_id'),
            ('flights', 'Flight', 'flight_id'),
            ('hotels', 'Hotel', 'hotel_id'),
            ('restaurants', 'Restaurant', 'restaurant_id'),
            ('preferences', 'Preference', 'preference_id'),
            ('users', 'User', 'User_ID'),
            ('passports', 'Passport', 'passport_id'),
            ('histories', 'History', 'history_id')
        ]
        
        # Process nodes in parallel
        print("Creating nodes...")
        node_start = time.time()
        for csv_name, label, id_field in node_config:
            if csv_name not in data or data[csv_name].empty:
                print(f"Skipping {label} nodes - no data available")
                continue
                
            # Prepare records
            records = [{"props": row.to_dict()} for _, row in data[csv_name].iterrows()]
            print(f"Creating {len(records)} {label} nodes...")
            
            # Batch create nodes
            self.neo4j.batch_create_nodes(label, id_field, records)
        
        print(f"Node creation completed in {time.time() - node_start:.2f} seconds")
        
        # Relationship processing
        print("Creating relationships...")
        rel_start = time.time()
        
        # Process relationships in parallel where possible
        if 'histories' in data and not data['histories'].empty:
            self.process_history_relationships(data)
        
        if 'preferences' in data and not data['preferences'].empty:
            self.process_preference_relationships(data)
        
        if 'passports' in data and not data['passports'].empty:
            self.process_passport_relationships(data)
        
        print(f"Relationship creation completed in {time.time() - rel_start:.2f} seconds")
        print(f"Graph build complete in {time.time() - start_time:.2f} seconds!")
    
    def process_history_relationships(self, data):
        print("Processing history relationships...")
        stayed_at_batch = []
        dined_at_batch = []
        
        for _, row in data['histories'].iterrows():
            hist_id = row.get('history_id')
            if pd.isna(hist_id):
                continue
            
            # STAYED_AT (History -> Hotel)
            if 'hotels' in row and isinstance(row['hotels'], str):
                try:
                    hotel_ids = ast.literal_eval(row['hotels'])
                    stayed_at_batch.extend({
                        "value_from": hist_id,
                        "value_to": h_id
                    } for h_id in hotel_ids)
                except:
                    pass
            
            # DINED_AT (History -> Restaurant)
            if 'restaurants' in row and isinstance(row['restaurants'], str):
                try:
                    rest_ids = ast.literal_eval(row['restaurants'])
                    dined_at_batch.extend({
                        "value_from": hist_id,
                        "value_to": r_id
                    } for r_id in rest_ids)
                except:
                    pass
        
        # Batch create relationships
        if stayed_at_batch:
            print(f"Creating {len(stayed_at_batch)} STAYED_AT relationships")
            self.neo4j.batch_create_relationships(
                "History", "history_id", "STAYED_AT", 
                "Hotel", "hotel_id", stayed_at_batch
            )
        
        if dined_at_batch:
            print(f"Creating {len(dined_at_batch)} DINED_AT relationships")
            self.neo4j.batch_create_relationships(
                "History", "history_id", "DINED_AT", 
                "Restaurant", "restaurant_id", dined_at_batch
            )
    
    def process_preference_relationships(self, data):
        print("Processing preference relationships...")
        hotel_pref_batch = []
        rest_pref_batch = []
        city_pref_batch = []
        
        for _, row in data['preferences'].iterrows():
            pref_id = row.get('preference_id')
            if pd.isna(pref_id):
                continue
            
            # HAS_HOTEL_PREFERENCE
            if 'top_hotels' in row and isinstance(row['top_hotels'], str):
                try:
                    hotel_ids = ast.literal_eval(row['top_hotels'])
                    hotel_pref_batch.extend({
                        "value_from": pref_id,
                        "value_to": h_id
                    } for h_id in hotel_ids)
                except:
                    pass
            
            # HAS_RESTAURANT_PREFERENCE
            if 'top_restaurants' in row and isinstance(row['top_restaurants'], str):
                try:
                    rest_ids = ast.literal_eval(row['top_restaurants'])
                    rest_pref_batch.extend({
                        "value_from": pref_id,
                        "value_to": r_id
                    } for r_id in rest_ids)
                except:
                    pass
            
            # HAS_CITY_PREFERENCE
            if 'top_cities' in row and isinstance(row['top_cities'], str):
                try:
                    city_names = ast.literal_eval(row['top_cities'])
                    city_pref_batch.extend({
                        "value_from": pref_id,
                        "value_to": cname
                    } for cname in city_names)
                except:
                    pass
        
        # Batch create relationships
        if hotel_pref_batch:
            print(f"Creating {len(hotel_pref_batch)} HAS_HOTEL_PREFERENCE relationships")
            self.neo4j.batch_create_relationships(
                "Preference", "preference_id", "HAS_HOTEL_PREFERENCE",
                "Hotel", "hotel_id", hotel_pref_batch
            )
        
        if rest_pref_batch:
            print(f"Creating {len(rest_pref_batch)} HAS_RESTAURANT_PREFERENCE relationships")
            self.neo4j.batch_create_relationships(
                "Preference", "preference_id", "HAS_RESTAURANT_PREFERENCE",
                "Restaurant", "restaurant_id", rest_pref_batch
            )
        
        if city_pref_batch:
            print(f"Creating {len(city_pref_batch)} HAS_CITY_PREFERENCE relationships")
            self.neo4j.batch_create_relationships(
                "Preference", "preference_id", "HAS_CITY_PREFERENCE",
                "City", "City", city_pref_batch
            )
    
    def process_passport_relationships(self, data):
        print("Processing passport relationships...")
        visa_pref_batch = []
        visa_history_batch = []
        
        for _, pport_row in data['passports'].iterrows():
            pport_id = pport_row.get('passport_id')
            if pd.isna(pport_id):
                continue
                
            origin = pport_row.get('Origin')
            
            # IS_READY_TO_APPLY_VISA (Preference -> Passport)
            if 'preferences' in data and 'visa_preference' in data['preferences'].columns:
                for _, pref_row in data['preferences'].iterrows():
                    if (pref_row.get('visa_preference') == pport_row.get('Requirement')):
                        visa_pref_batch.append({
                            "value_from": pref_row['preference_id'],
                            "value_to": pport_id
                        })
            
            # REQUIRED_VISA_LIKE (Passport -> History)
            if origin and 'histories' in data:
                for _, hist_row in data['histories'].iterrows():
                    if hist_row.get('issued_passport') == origin:
                        visa_history_batch.append({
                            "value_from": pport_id,
                            "value_to": hist_row['history_id']
                        })
        
        # Batch create relationships
        if visa_pref_batch:
            print(f"Creating {len(visa_pref_batch)} IS_READY_TO_APPLY_VISA relationships")
            self.neo4j.batch_create_relationships(
                "Preference", "preference_id", "IS_READY_TO_APPLY_VISA",
                "Passport", "passport_id", visa_pref_batch
            )
        
        if visa_history_batch:
            print(f"Creating {len(visa_history_batch)} REQUIRED_VISA_LIKE relationships")
            self.neo4j.batch_create_relationships(
                "Passport", "passport_id", "REQUIRED_VISA_LIKE",
                "History", "history_id", visa_history_batch
            )

#########################################
# 3. Representation Agent (Optimized)
#########################################
class RepresentationAgent:
    @staticmethod
    def represent_node(node, fields):
        parts = []
        props = node._properties
        for field, label in fields.items():
            if field in props and pd.notna(props[field]):
                parts.append(f"{label}: {props[field]}")
        return "; ".join(parts)
    
    @classmethod
    def get_all_representations(cls, neo4j_agent):
        print("Generating node representations...")
        start_time = time.time()
        
        all_nodes = []
        label_config = [
            ("City", {
                "City": "City", "Country": "Country",
                "Remote connection: Average WiFi speed (Mbps per second)": "WiFi Speed"
            }),
            ("Flight", {
                "Airline": "Airline", "Total Fare (EUR)": "Price",
                "Departure Airport Code": "From", "Arrival Airport Code": "To"
            }),
            ("Hotel", {
                "name": "Name", "price": "Price",
                "number_reviews": "Reviews", "City": "City"
            }),
            ("Restaurant", {
                "Restaurant Name": "Name", "Cuisines": "Cuisines",
                "Average Cost for two": "Price for Two", "City": "City"
            }),
            ("Preference", {"preference_id": "Preference ID"}),
            ("User", {"User_ID": "User ID", "user_id": "User ID"}),
            ("Passport", {"passport_id": "Passport ID", "Origin": "Origin"}),
            ("History", {"history_id": "History ID", "city": "City"})
        ]
        
        # Process labels in parallel with progress tracking
        with ThreadPoolExecutor() as executor:
            futures = []
            for label, fields in label_config:
                futures.append(executor.submit(
                    cls.process_label,
                    neo4j_agent, label, fields
                ))
            
            for future in as_completed(futures):
                nodes = future.result()
                all_nodes.extend(nodes)
                print(f"Processed {len(nodes)} {nodes[0].split(':')[0] if nodes else ''} representations")
        
        print(f"Generated {len(all_nodes)} representations in {time.time() - start_time:.2f} seconds")
        return all_nodes
    
    @classmethod
    def process_label(cls, neo4j_agent, label, fields):
        nodes = []
        result = neo4j_agent.run_query(f"MATCH (n:{label}) RETURN n", use_cache=True)
        for record in result:
            rep = cls.represent_node(record["n"], fields)
            if rep:
                nodes.append(rep)
        return nodes

#########################################
# 4. Embedding Agent (Optimized)
#########################################
class EmbeddingAgent:
    def __init__(self):
        # Use a smaller, faster model
        self.embedder = SentenceTransformer("all-MiniLM-L6-v2", device='cpu')
        self.doc_embeddings = None
        self.documents = []
    
    def update_documents(self, documents):
        print("Updating document embeddings...")
        start_time = time.time()
        
        self.documents = documents
        batch_size = 256  # Larger batch size for faster processing
        
        # Process embeddings in parallel batches
        embeddings = []
        for i in range(0, len(documents), batch_size):
            batch = documents[i:i + batch_size]
            embeddings.append(self.embedder.encode(batch, convert_to_tensor=True))
        
        self.doc_embeddings = torch.cat(embeddings) if embeddings else None
        print(f"Embeddings updated in {time.time() - start_time:.2f} seconds")
    
    def retrieve_documents(self, query, top_k=8):
        if self.doc_embeddings is None or len(self.doc_embeddings) == 0:
            return []
            
        # Cache query results
        cache_key = hash(query)
        if hasattr(self, '_cache') and cache_key in self._cache:
            return self._cache[cache_key]
            
        query_embedding = self.embedder.encode([query], convert_to_tensor=True)
        
        # Use FAISS for faster similarity search if available
        try:
            import faiss
            if not hasattr(self, 'faiss_index'):
                # Create FAISS index if it doesn't exist
                d = self.doc_embeddings.shape[1]
                self.faiss_index = faiss.IndexFlatIP(d)
                self.faiss_index.add(self.doc_embeddings.cpu().numpy())
            
            D, I = self.faiss_index.search(query_embedding.cpu().numpy(), top_k)
            results = [self.documents[i] for i in I[0]]
        except ImportError:
            # Fall back to cosine similarity if FAISS not available
            cos_scores = cosine_similarity(
                query_embedding.cpu().numpy(), 
                self.doc_embeddings.cpu().numpy()
            )[0]
            top_indices = np.argpartition(cos_scores, -top_k)[-top_k:]
            top_indices = top_indices[np.argsort(cos_scores[top_indices])[::-1]]
            results = [self.documents[i] for i in top_indices]
        
        # Cache results
        if not hasattr(self, '_cache'):
            self._cache = {}
        self._cache[cache_key] = results
        
        return results

#########################################
# 5. Query Agent (Optimized)
#########################################
class QueryAgent:
    def __init__(self):
        # Use a smaller, faster model for query refinement
        from transformers import pipeline
        self.generator = pipeline(
            "text-generation",
            model="distilgpt2",  # Smaller than GPT-2
            device="cpu",
            max_length=50
        )
    
    def detect_query_type(self, query):
        query = query.lower()
        if any(w in query for w in ["hotel", "stay", "accommodation"]):
            return "hotel"
        elif any(w in query for w in ["restaurant", "eat", "dine", "food"]):
            return "restaurant"
        elif any(w in query for w in ["flight", "fly", "airline"]):
            return "flight"
        elif any(w in query for w in ["city", "destination", "place", "wifi", "digital nomad"]):
            return "city"
        elif any(w in query for w in ["trip", "itinerary", "plan"]):
            return "complete_trip"
        elif any(w in query for w in ["clear", "reset"]):
            return "clear_history"
        return "general"
    
    def refine_query(self, raw_query, query_type):
        if query_type == "clear_history":
            return raw_query
            
        # Simple refinement for common cases to avoid model call
        if query_type == "hotel":
            if "under" in raw_query.lower() and "$" in raw_query:
                return f"Find hotels {raw_query}"
            return f"Find hotels in {raw_query}"
        elif query_type == "restaurant":
            return f"Find restaurants in {raw_query}"
        elif query_type == "flight":
            return f"Find flights to {raw_query}"
        elif query_type == "city":
            return f"Find cities matching {raw_query}"
        
        # Fall back to model for complex queries
        prompt = f"Make this travel query more specific for {query_type} search: {raw_query}"
        result = self.generator(prompt, max_length=50, do_sample=False)
        return result[0]["generated_text"].replace(prompt, "").strip()

#########################################
# 6. Response Agent (Optimized)
#########################################
class ResponseAgent:
    def __init__(self, neo4j_agent, embedding_agent, query_agent):
        self.neo4j = neo4j_agent
        self.embedding = embedding_agent
        self.query = query_agent
        self.cached_cities = None
        self.cached_countries = None
        self.query_cache = {}
    
    def get_cities(self):
        if self.cached_cities is None:
            result = self.neo4j.run_query("MATCH (c:City) RETURN c.City as city", use_cache=True)
            self.cached_cities = [r["city"] for r in result if r["city"]]
        return self.cached_cities
    
    def get_countries(self):
        if self.cached_countries is None:
            result = self.neo4j.run_query("MATCH (c:City) RETURN c.Country as country", use_cache=True)
            self.cached_countries = [r["country"] for r in result if r["country"]]
        return self.cached_countries
    
    def query_neo4j(self, query_type, filters=None):
        if not filters:
            filters = {}
        
        # Cache key for query results
        cache_key = (query_type, frozenset(filters.items()))
        if cache_key in self.query_cache:
            return self.query_cache[cache_key]
        
        queries = {
            "hotel": """
                MATCH (h:Hotel)
                WHERE ($city IS NULL OR h.City = $city)
                AND ($max_price IS NULL OR toFloat(h.price) <= $max_price)
                RETURN h ORDER BY toFloat(h.price) LIMIT 5
            """,
            "restaurant": """
                MATCH (r:Restaurant)
                WHERE ($city IS NULL OR r.City = $city)
                AND ($cuisine IS NULL OR toLower(r.Cuisines) CONTAINS toLower($cuisine))
                RETURN r ORDER BY toFloat(r.`Average Cost for two`) LIMIT 5
            """,
            "flight": """
                MATCH (f:Flight)
                WHERE ($destination IS NULL OR toLower(f.`Arrival Airport Code`) = toLower($destination))
                AND ($max_price IS NULL OR toFloat(f.`Total Fare (EUR)`) <= $max_price)
                RETURN f ORDER BY toFloat(f.`Total Fare (EUR)`) LIMIT 5
            """,
            "city": """
                MATCH (c:City)
                WHERE ($country IS NULL OR toLower(c.Country) = toLower($country))
                AND ($wifi_speed IS NULL OR toFloat(c.`Remote connection: Average WiFi speed (Mbps per second)`) >= $wifi_speed)
                RETURN c LIMIT 5
            """
        }
        
        if query_type not in queries:
            return []
        
        # Set default parameters to avoid missing parameter errors
        if query_type == "flight":
            filters.setdefault("destination", None)
            filters.setdefault("max_price", None)
        elif query_type == "hotel":
            filters.setdefault("city", None)
            filters.setdefault("max_price", None)
        elif query_type == "restaurant":
            filters.setdefault("city", None)
            filters.setdefault("cuisine", None)
        elif query_type == "city":
            filters.setdefault("country", None)
            filters.setdefault("wifi_speed", None)
        
        try:
            result = self.neo4j.run_query(queries[query_type], filters, use_cache=True)
            nodes = [record["h" if query_type == "hotel" else "r" if query_type == "restaurant" 
                    else "f" if query_type == "flight" else "c"] for record in result]
            
            # Cache results
            self.query_cache[cache_key] = nodes
            return nodes
        except Exception as e:
            print(f"Error querying Neo4j: {e}")
            return []
    
    def extract_days(self, query):
        """Extract number of days from trip query"""
        match = re.search(r'(\d+)\s*day', query.lower())
        return int(match.group(1)) if match else 5  # Default to 5 days
    
    def generate_response(self, query):
        # Check cache first
        if query in self.query_cache:
            return self.query_cache[query]
            
        query_type = self.query.detect_query_type(query)
        refined = self.query.refine_query(query, query_type)
        
        if query_type == "clear_history":
            response = "Conversation cleared. How can I help with your travel plans?"
            self.query_cache[query] = (response, [])
            return response, []
        
        # Extract filters
        filters = {}
        if query_type == "hotel":
            for city in self.get_cities():
                if city and city.lower() in refined.lower():
                    filters["city"] = city
                    break
            if "under" in refined.lower() and "$" in refined:
                try:
                    filters["max_price"] = float(refined.split("$")[1].split()[0])
                except:
                    pass
        
        elif query_type == "restaurant":
            for city in self.get_cities():
                if city and city.lower() in refined.lower():
                    filters["city"] = city
                    break
            cuisines = ["italian", "chinese", "french", "japanese", "mexican"]
            for cuisine in cuisines:
                if cuisine in refined.lower():
                    filters["cuisine"] = cuisine
                    break
        
        elif query_type == "flight":
            for city in self.get_cities():
                if city and city.lower() in refined.lower():
                    filters["destination"] = city
                    break
            if "under" in refined.lower() and "$" in refined:
                try:
                    filters["max_price"] = float(refined.split("$")[1].split()[0])
                except:
                    pass
        
        elif query_type == "city":
            for country in self.get_countries():
                if country and country.lower() in refined.lower():
                    filters["country"] = country
                    break
            if "wifi" in refined.lower() or "internet" in refined.lower() or "digital nomad" in refined.lower():
                filters["wifi_speed"] = 50
        
        # Get results
        neo4j_results = self.query_neo4j(query_type, filters)
        retrieved_docs = self.embedding.retrieve_documents(refined)
        
        # Generate response
        if query_type == "hotel":
            if not neo4j_results:
                response = "I couldn't find any hotels matching your criteria in our database."
            else:
                response = "Here are some hotel recommendations:\n"
                for hotel in neo4j_results:
                    props = hotel._properties
                    response += f"- {props.get('name')} (${props.get('price')}, {props.get('rating', '?')}★)\n"
        
        elif query_type == "restaurant":
            if not neo4j_results:
                response = "I couldn't find any restaurants matching your criteria in our database."
            else:
                response = "Here are some restaurant recommendations:\n"
                for rest in neo4j_results:
                    props = rest._properties
                    response += f"- {props.get('Restaurant Name')} ({props.get('Cuisines')}, ${props.get('Average Cost for two')})\n"
        
        elif query_type == "flight":
            if not neo4j_results:
                response = "I couldn't find any flight information for this destination in our database."
            else:
                response = "Here are some flight options:\n"
                for flight in neo4j_results:
                    props = flight._properties
                    response += f"- {props.get('Airline')} from {props.get('Departure Airport Code')} to {props.get('Arrival Airport Code')} (${props.get('Total Fare (EUR)')})\n"
        
        elif query_type == "city":
            if not neo4j_results:
                response = "I couldn't find any cities matching your criteria in our database."
            else:
                response = "Here are some city recommendations:\n"
                for city in neo4j_results:
                    props = city._properties
                    response += f"- {props.get('City')}, {props.get('Country')} (WiFi: {props.get('Remote connection: Average WiFi speed (Mbps per second)')}Mbps)\n"
        
        elif query_type == "complete_trip":
            # Get destination city from query
            destination = None
            for city in self.get_cities():
                if city and city.lower() in query.lower():
                    destination = city
                    break
            
            if not destination:
                response = "Please specify a destination city for trip planning."
            else:
                days = self.extract_days(query)
                response = f"Here's a suggested {days}-day trip plan for {destination}:\n\n"
                
                # Get flights
                flights = self.query_neo4j("flight", {"destination": destination})
                if flights:
                    response += f"Flight options to {destination}:\n"
                    for flight in flights[:2]:  # Show top 2 options
                        props = flight._properties
                        response += f"- {props.get('Airline')} from {props.get('Departure Airport Code')} (${props.get('Total Fare (EUR)')})\n"
                    response += "\n"
                else:
                    response += f"Sorry, I couldn't find any flight information for {destination} in our database.\n\n"
                
                # Get hotels with budget consideration
                hotel_filters = {"city": destination}
                if "under" in query.lower() and "$" in query:
                    try:
                        max_price = float(query.split("$")[1].split()[0])
                        hotel_filters["max_price"] = max_price
                        response += f"Hotels under ${max_price}:\n"
                    except:
                        response += "Recommended hotels:\n"
                else:
                    response += "Recommended hotels:\n"
                
                hotels = self.query_neo4j("hotel", hotel_filters)
                if hotels:
                    for hotel in hotels[:3]:  # Show top 3 options
                        props = hotel._properties
                        response += f"- {props.get('name')} (${props.get('price')}/night, {props.get('rating', '?')}★)\n"
                    response += "\n"
                else:
                    response += f"Sorry, I couldn't find any hotel information for {destination} in our database.\n\n"
                
                # Get restaurants with cuisine consideration
                restaurant_filters = {"city": destination}
                cuisines = ["italian", "chinese", "french", "japanese", "mexican"]
                for cuisine in cuisines:
                    if cuisine in query.lower():
                        restaurant_filters["cuisine"] = cuisine
                        response += f"{cuisine.capitalize()} restaurants:\n"
                        break
                else:
                    response += "Recommended restaurants:\n"
                
                restaurants = self.query_neo4j("restaurant", restaurant_filters)
                if restaurants:
                    for rest in restaurants[:5]:  # Show more restaurants for multi-day trip
                        props = rest._properties
                        response += f"- {props.get('Restaurant Name')} ({props.get('Cuisines')}, ${props.get('Average Cost for two')})\n"
                    response += "\n"
                else:
                    response += f"Sorry, I couldn't find any restaurant information for {destination} in our database.\n\n"
                
                # Add daily itinerary suggestions
                response += f"Suggested {days}-day itinerary:\n"
                for day in range(1, days+1):
                    response += f"\nDay {day}:\n"
                    response += "- Morning: Explore local attractions\n"
                    response += "- Lunch: Try a local restaurant\n"
                    response += "- Afternoon: Visit cultural sites\n"
                    response += "- Dinner: Enjoy local cuisine\n"
                
                response += "\nNote: This is a suggested itinerary based on available data. Please verify details before travel."
        
        else:
            from transformers import pipeline
            generator = pipeline(
                "text-generation",
                model="distilgpt2",  # Smaller model
                device="cpu",
                max_length=150
            )
            prompt = f"Answer this travel question: {query}\nContext: {retrieved_docs[:3]}"
            result = generator(prompt, max_length=150)
            response = result[0]["generated_text"].replace(prompt, "").strip()
        
        # Cache response
        self.query_cache[query] = (response, [str(r) for r in neo4j_results[:5]])
        return response, [str(r) for r in neo4j_results[:5]]

#########################################
# 7. Travel Assistant Agent (Optimized)
#########################################
class TravelAssistantAgent:
    def __init__(self, neo4j_uri, neo4j_user, neo4j_password, csv_paths):
        self.neo4j = Neo4jAgent(neo4j_uri, neo4j_user, neo4j_password)
        self.graph_builder = GraphBuilderAgent(self.neo4j, csv_paths)
        self.embedding = EmbeddingAgent()
        self.query = QueryAgent()
        self.response = ResponseAgent(self.neo4j, self.embedding, self.query)
        self.history = []
        
        # Build graph and embeddings (in parallel)
        print("\nInitializing Travel Assistant...")
        start_time = time.time()
        
        print("Building graph...")
        self.graph_builder.build_graph()
        
        print("Generating representations...")
        docs = RepresentationAgent.get_all_representations(self.neo4j)
        
        print("Creating embeddings...")
        self.embedding.update_documents(docs)
        
        print(f"\nTravel Assistant ready in {time.time() - start_time:.2f} seconds!")
    
    def handle_query(self, query):
        answer, data = self.response.generate_response(query)
        self.history.append({"sender": "User", "text": query})
        self.history.append({"sender": "Assistant", "text": answer})
        return answer, data

#########################################
# 8. Flask Web Interface (Optimized)
#########################################
app = Flask(__name__)

HTML_TEMPLATE = """
<!DOCTYPE html>
<html>
<head>
    <title>Neo4j Travel Assistant</title>
    <style>
        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            max-width: 1200px;
            margin: 0 auto;
            padding: 20px;
            background-color: #f5f7fa;
            color: #333;
        }
        .header {
            background-color: #4285f4;
            color: white;
            padding: 20px;
            border-radius: 8px;
            margin-bottom: 20px;
            text-align: center;
        }
        .filter-section {
            background-color: white;
            padding: 15px;
            border-radius: 8px;
            margin-bottom: 20px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }
        .filter-buttons {
            display: flex;
            gap: 10px;
            flex-wrap: wrap;
            margin-top: 10px;
        }
        .filter-button {
            padding: 8px 15px;
            background-color: #e0e0e0;
            border: none;
            border-radius: 20px;
            cursor: pointer;
            transition: background-color 0.3s;
        }
        .filter-button:hover {
            background-color: #d0d0d0;
        }
        .filter-button.active {
            background-color: #4285f4;
            color: white;
        }
        .chat-container {
            background-color: white;
            border-radius: 8px;
            padding: 20px;
            margin-bottom: 20px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            height: 500px;
            overflow-y: auto;
            white-space: pre-wrap;
        }
        .message {
            margin-bottom: 15px;
            padding: 10px 15px;
            border-radius: 18px;
            max-width: 80%;
            word-wrap: break-word;
        }
        .user-message {
            background-color: #e3f2fd;
            margin-left: auto;
            border-bottom-right-radius: 4px;
        }
        .bot-message {
            background-color: #f1f1f1;
            margin-right: auto;
            border-bottom-left-radius: 4px;
            white-space: pre-wrap;
        }
        .data-section {
            background-color: white;
            border-radius: 8px;
            padding: 20px;
            margin-bottom: 20px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            max-height: 300px;
            overflow-y: auto;
        }
        .data-item {
            padding: 10px;
            border-bottom: 1px solid #eee;
            font-family: monospace;
        }
        .input-section {
            display: flex;
            gap: 10px;
        }
        #user-input {
            flex-grow: 1;
            padding: 12px;
            border: 1px solid #ddd;
            border-radius: 8px;
            font-size: 16px;
        }
        #submit-button {
            padding: 12px 20px;
            background-color: #4285f4;
            color: white;
            border: none;
            border-radius: 8px;
            cursor: pointer;
            font-size: 16px;
        }
        #submit-button:hover {
            background-color: #3367d6;
        }
        .query-type-indicator {
            font-size: 14px;
            color: #666;
            margin-top: 5px;
            font-style: italic;
        }

        .neo4j-info {
            background-color: #008cc1;
            color: white;
            padding: 10px;
            border-radius: 5px;
            margin-top: 10px;
            font-size: 14px;
        }
    </style>
</head>
<body>
    <div class="header">
        <h1>Neo4j Travel Assistant</h1>
        <p>Get personalized travel recommendations powered by Neo4j graph database</p>
        <div class="neo4j-info">
            Connected to Neo4j at {{ NEO4J_URI }} with {{ node_count }} nodes in database
        </div>
    </div>
    
    <div class="filter-section">
        <h3>Not sure what to ask? Try these Neo4j-powered queries:</h3>
        <div class="filter-buttons">
            <button class="filter-button" onclick="setQuery('Best hotels in London under $200')">Hotels</button>
            <button class="filter-button" onclick="setQuery('Italian Restaurants in London')">Restaurants</button>
            <button class="filter-button" onclick="setQuery('Cheapest flights to Paris')">Flights</button>
            <button class="filter-button" onclick="setQuery('Best digital nomad cities with good WiFi')">Destinations</button>
            <button class="filter-button" onclick="setQuery('Plan a complete 5-day trip to Istanbul')">Complete Trip</button>
        </div>
    </div>
    
    <div class="chat-container" id="chat-container">
        {% for msg in history %}
            <div class="message {% if msg.sender == 'User' %}user-message{% else %}bot-message{% endif %}">
                <strong>{{ msg.sender }}:</strong> {{ msg.text }}
                {% if msg.query_type %}
                <div class="query-type-indicator">Detected as: {{ msg.query_type }}</div>
                {% endif %}
            </div>
        {% endfor %}
    </div>
    
    <div class="data-section">
        <h3>Neo4j Data Details</h3>
        {% if retrieved_data %}
            {% for doc in retrieved_data %}
                <div class="data-item">{{ doc }}</div>
            {% endfor %}
        {% else %}
            <div class="data-item">No data retrieved yet. Ask about hotels, restaurants, flights or destinations.</div>
        {% endif %}
    </div>
    
    <form method="post" class="input-section">
        <input type="text" id="user-input" name="question" placeholder="Ask about hotels, flights, restaurants or destinations..." required>
        <input type="submit" id="submit-button" value="Send">
    </form>
    
    <script>
        function setQuery(query) {
            document.getElementById('user-input').value = query;
            if (query.toLowerCase().includes('clear')) {
                document.forms[0].submit();
            }
            document.getElementById('user-input').focus();
        }
        
        // Auto-scroll chat to bottom
        var chatContainer = document.getElementById("chat-container");
        chatContainer.scrollTop = chatContainer.scrollHeight;
    </script>
</body>
</html>
"""

# Define CSV file paths
CSV_PATHS = {
    "cities": "adjusted_datasets/adjusted_cities.csv",
    "flights": "adjusted_datasets/adjusted_flights.csv",
    "hotels": "adjusted_datasets/adjusted_hotels.csv",
    "restaurants": "adjusted_datasets/adjusted_restaurants.csv",
    "preferences": "adjusted_datasets/preferences.csv",
    "users": "adjusted_datasets/users.csv",
    "passports": "adjusted_datasets/adjusted_passports.csv",
    "histories": "adjusted_datasets/histories.csv"
}

# Initialize the Travel Assistant Agent
print("Starting application...")
assistant = TravelAssistantAgent("bolt://localhost:7687", "neo4j", "argentic", CSV_PATHS)

@app.route("/", methods=["GET", "POST"])
def index():
    retrieved_data = []
    node_count = assistant.neo4j.get_node_count()
    if request.method == "POST":
        question = request.form["question"]
        start_time = time.time()
        answer, retrieved_data = assistant.handle_query(question)
        print(f"Query processed in {time.time() - start_time:.2f} seconds")
    return render_template_string(
        HTML_TEMPLATE,
        history=assistant.history,
        retrieved_data=retrieved_data,
        NEO4J_URI=assistant.neo4j.uri,
        node_count=node_count
    )

if __name__ == "__main__":
    app.run(port=5001, debug=True, use_reloader=False)

Starting application...
Connected to Neo4j successfully.


Device set to use cpu



Initializing Travel Assistant...
Building graph...
Starting graph build...
Loading CSV files...
Successfully loaded adjusted_datasets/adjusted_cities.csv with 22 rows
Successfully loaded adjusted_datasets/adjusted_hotels.csv with 96 rows
Successfully loaded adjusted_datasets/preferences.csv with 152 rows
Successfully loaded adjusted_datasets/users.csv with 200 rows
Successfully loaded adjusted_datasets/adjusted_restaurants.csv with 228 rows
Successfully loaded adjusted_datasets/adjusted_passports.csv with 398 rows
Successfully loaded adjusted_datasets/histories.csv with 761 rows
Successfully loaded adjusted_datasets/adjusted_flights.csv with 11979 rows
CSV loading completed in 0.16 seconds
Creating nodes...
Creating 22 City nodes...
Creating 11979 Flight nodes...
Creating 96 Hotel nodes...
Creating 228 Restaurant nodes...
Creating 152 Preference nodes...
Creating 200 User nodes...
Creating 398 Passport nodes...
Creating 761 History nodes...
Node creation completed in 28.69 seconds
Cre

 * Running on http://127.0.0.1:5001
Press CTRL+C to quit
127.0.0.1 - - [29/Mar/2025 22:12:02] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [29/Mar/2025 22:12:05] "POST / HTTP/1.1" 200 -


Query processed in 0.09 seconds


...

Query processed in 2.06 seconds


127.0.0.1 - - [29/Mar/2025 22:12:55] "POST / HTTP/1.1" 200 -


Query processed in 0.09 seconds


127.0.0.1 - - [29/Mar/2025 22:14:20] "POST / HTTP/1.1" 200 -


Query processed in 0.09 seconds


# Analyse des Codes: Neo4j-basierter Reiseassistent

## Überblick der Architektur

Der Code implementiert einen Reiseassistenten, der auf einer Neo4j-Graphdatenbank basiert und verschiedene KI-Methoden integriert. Die Hauptkomponenten sind:

1. **Neo4j Connection Agent**: Verwaltet die Verbindung zur Graphdatenbank
2. **Graph Builder Agent**: Erstellt den Graphen aus CSV-Daten
3. **Representation Agent**: Erstellt Textrepräsentationen der Knoten
4. **Embedding Agent**: Erzeugt Embeddings für semantische Suche
5. **Query Agent**: Analysiert und verfeinert Benutzeranfragen
6. **Response Agent**: Generiert Antworten basierend auf Daten
7. **Travel Assistant Agent**: Koordiniert die Hauptlogik
8. **Flask Web Interface**: Stellt die Benutzeroberfläche bereit

## Methoden und Modelle

### 1. Neo4j-Interaktion
- **Batch-Verarbeitung**: Verwendet Batch-Operationen für effiziente Datenbankzugriffe
- **ThreadPoolExecutor**: Parallele Ausführung von Datenbankoperationen
- **Caching**: Häufige Abfragen werden zwischengespeichert

### 2. Embedding-Modell
- **SentenceTransformer**: Verwendet `all-MiniLM-L6-v2` statt GPT-2
- **FAISS-Integration**: Für effiziente Ähnlichkeitssuche (falls verfügbar)
- **Cosine Similarity**: Fallback wenn FAISS nicht verfügbar

### 3. Query-Verarbeitung
- **DistilGPT-2**: Kleinere Version von GPT-2 für Query-Verfeinerung
- **Regelbasierte Klassifikation**: Erkennt Query-Typen (Hotel, Restaurant etc.)
- **Cache-Mechanismus**: Speichert vorherige Anfragen und Antworten

## Warum nicht GPT-2?

1. **Performance**: DistilGPT-2 und MiniLM sind kleiner und schneller
2. **Ressourcenverbrauch**: Laufen auf CPU statt GPU
3. **Spezialisierung**: Für diese Anwendung ausreichend
4. **Kosten**: Keine API-Kosten wie bei größeren Modellen

## Warum kein LLaMA?
1. **Größe**: LLaMA selbst (ab 7B Parametern) wäre für diese Anwendung überdimensioniert.
2. **Hardware-Anforderungen**: LLaMA benötigt typischerweise GPUs/TPUs für akzeptable Performance.
3. **Use-Case**: Die Anwendung setzt primär auf strukturierte Neo4j-Abfragen + leichte Embeddings, nicht auf freie Textgenerierung.

## State-of-the-Art Alternativen

1. **Embedding-Modelle**:
   - `all-MiniLM-L6-v2`: Guter Kompromiss zwischen Größe und Leistung
   - Alternativen: `multi-qa-mpnet-base-dot-v1` (besser für Q&A)

2. **Generative Modelle**:
   - DistilGPT-2: Leichtgewichtige Version von GPT-2
   - Neuere Optionen: GPT-3/4, Falcon, LLaMA (wären leistungsfähiger aber ressourcenintensiver)

## Komponenten-Struktur

```
Flask Web Interface
│
└── TravelAssistantAgent (Koordinator)
    ├── Neo4jAgent (Datenbankverbindung)
    ├── GraphBuilderAgent (Grapherstellung)
    ├── RepresentationAgent (Knotenrepräsentation)
    ├── EmbeddingAgent (Semantische Suche)
    ├── QueryAgent (Anfrageanalyse)
    └── ResponseAgent (Antwortgenerierung)
```

## Datenfluss

1. Benutzeranfrage → Flask Interface
2. → TravelAssistantAgent.handle_query()
3. → QueryAgent.detect_query_type() + refine_query()
4. → ResponseAgent.generate_response()
   - Neo4j-Abfragen für strukturierte Daten
   - Embedding-Suche für semantische Ähnlichkeit
5. → Kombinierte Antwort an Benutzer

## Optimierungen im Code

1. **Parallelisierung**: ThreadPoolExecutor für Datenbankoperationen
2. **Batch-Verarbeitung**: Effiziente Datenbankzugriffe
3. **Caching**: Häufige Abfragen werden gespeichert
4. **Leichtgewichtige Modelle**: Für CPU-Einsatz optimiert
5. **FAISS-Integration**: Beschleunigte Vektorsuche

## Beispiel-Interaktionen

Der Code zeigt typische Anfragen wie:
- Hotelsuche ("Best hotels in London under $200")
- Restaurantempfehlungen ("Italian Restaurants in London")
- Flugsuche ("Cheapest flights to Paris")
- Reiseplanung ("Plan a complete 5-day trip to Istanbul")

Die Antworten kombinieren strukturierte Neo4j-Daten mit generiertem Text.